# 01 — Data Loading and Link Utilisation

**Goal:** turn the raw Abilene backbone measurements into a table of
*how busy every link was, at every 5-minute interval, over six months*.

The network is the Internet2 Abilene backbone (US research network), measured
March–September 2004. Three ingredients:

| Ingredient | File | What it gives us |
|---|---|---|
| Topology | `topo-2003-04-10.txt` | 12 routers, 30 directed links, **real capacities** and OSPF weights |
| Routing matrix | `A` | which links each origin→destination flow actually crossed |
| Traffic matrices | `X01.gz` … `X24.gz` | how much traffic each origin→destination pair sent, every 5 minutes |

Utilisation is then simply:

$$\text{utilisation} = \frac{\text{traffic routed onto the link}}{\text{link capacity}}$$

**One important choice:** we do *not* compute shortest paths ourselves. The dataset
ships the network's own OSPF-derived routing matrix `A`, so we read how traffic
really flowed instead of assuming how it should have flowed.

In [1]:
from pathlib import Path
import gzip

import numpy as np
import pandas as pd

# This notebook lives in notebooks/, so the project root is one level up.
PROJECT = Path.cwd().parent
RAW = PROJECT / "data" / "raw" / "abilene"
PROCESSED = PROJECT / "data" / "processed"

pd.set_option("display.width", 120)

print("reading from:", RAW)
print("files:", sorted(p.name for p in RAW.iterdir()))

reading from: C:\NETWORK-CONGESTION\TRAFFIC-CONGESTION-PREDICTION\data\raw\abilene
files: ['A', 'demands', 'links', 'readme.txt', 'topo-2003-04-10.txt', 'traffic-matrices']


## Step 1 — Topology: routers, capacities, OSPF weights

`topo-2003-04-10.txt` has two sections, one after the other:

```
router
#name   city         latitude   longitude
ATLAng  Atlanta_GA   33.750000  -84.383300
...
link
#x      y        capacity(kbps)  OSPF
ATLAng  HSTNng   9920000         1176
```

So we read it line by line and keep track of which section we are in.

In [2]:
routers = []
links = []
section = None

for line in (RAW / "topo-2003-04-10.txt").read_text().splitlines():
    line = line.strip()

    if line in ("router", "link"):
        section = line          # a section header - remember where we are
        continue
    if line == "" or line.startswith("#"):
        continue                # blank line or comment

    parts = line.split()
    if section == "router":
        # name  city  latitude  longitude
        routers.append({"node": parts[0], "city": parts[1],
                        "lat": float(parts[2]), "lon": float(parts[3])})
    else:
        # source  target  capacity(kbps)  ospf_weight
        links.append({"src": parts[0], "dst": parts[1],
                      "capacity_kbps": float(parts[2]), "ospf": float(parts[3])})

routers = pd.DataFrame(routers)
links = pd.DataFrame(links)

# A single string name per directed link, e.g. "CHINng,IPLSng".
# This is the same format the `links` index file uses, so they join cleanly later.
links["link"] = links["src"] + "," + links["dst"]

print(f"{len(routers)} routers, {len(links)} directed links")
routers

12 routers, 30 directed links


,node,city,lat,lon
0,ATLA-M5,Atlanta_GA,33.750000,-84.383300
1,ATLAng,Atlanta_GA,33.750000,-84.383300
2,CHINng,Chicago_IL,41.833300,-87.616700
3,DNVRng,Denver_CO,40.750000,-105.000000
4,HSTNng,Houston_TX,29.770031,-95.517364
5,IPLSng,Indianapolis_IN,39.780622,-86.159535
6,KSCYng,Kansas_City_MO,38.961694,-96.596704
7,LOSAng,Los_Angeles_CA,34.050000,-118.250000
8,NYCMng,New_York_NY,40.783300,-73.966700
9,SNVAng,Sunnyvale_CA,37.385750,-122.025530


### The capacities are real

This matters. An earlier attempt at this project used a different copy of Abilene
whose topology file listed **1 Gbps** links — a figure from a network *simulator*,
not the real backbone. Utilisation came out below 1% on every link and there was
no meaningful variation to work with.

These are the genuine provisioned capacities: **OC-192 = 9.92 Gbps** on the
backbone, with one **OC-48 = 2.48 Gbps** pair between Atlanta and Indianapolis.

In [3]:
print("capacity (kbps) -> number of links")
print(links["capacity_kbps"].value_counts())

print("\nThe two lower-capacity (OC-48) links:")
links[links["capacity_kbps"] < 9_920_000][["link", "capacity_kbps", "ospf"]]

capacity (kbps) -> number of links
capacity_kbps
9920000.0    28
2480000.0     2
Name: count, dtype: int64

The two lower-capacity (OC-48) links:


,link,capacity_kbps,ospf
3,"ATLAng,IPLSng",2480000.0,587.0
13,"IPLSng,ATLAng",2480000.0,587.0


## Step 2 — The link and demand index files

The routing matrix refers to links and origin–destination (OD) pairs by *number*,
not by name. Two small lookup files translate:

- `links` — `"SRC,DST"` → link number (1–54), plus a type
  (**0 = internal backbone**, 1 = traffic entering from outside, 2 = leaving)
- `demands` — `"SRC,DST"` → OD pair number (1–144)

We only care about the 30 **internal** links: those are the backbone links that
have a capacity and that we can reroute traffic over.

In [4]:
link_index = pd.read_csv(RAW / "links", sep=r"\s+", comment="#",
                         names=["link", "link_idx", "link_type"])
demand_index = pd.read_csv(RAW / "demands", sep=r"\s+", comment="#",
                           names=["od", "demand_idx"])

# Keep only internal backbone links (type 0), in their original index order.
internal = link_index[link_index["link_type"] == 0].reset_index(drop=True)

print(f"{len(link_index)} links total -> {len(internal)} internal backbone links")
print(f"{len(demand_index)} OD pairs  (12 routers x 12 = 144)")
internal.head()

54 links total -> 30 internal backbone links
144 OD pairs  (12 routers x 12 = 144)


,link,link_idx,link_type
0,"ATLA-M5,ATLAng",1,0
1,"ATLAng,ATLA-M5",2,0
2,"ATLAng,HSTNng",3,0
3,"ATLAng,IPLSng",4,0
4,"ATLAng,WASHng",5,0


## Step 3 — The routing matrix `A`

`A` is the heart of this notebook. Each line says:

> *this fraction of the traffic for this OD pair travels over this link*

For example, traffic from Seattle to Washington DC crosses several links on the
way; `A` lists each one. So to get the load on a link, we add up the demand of
every OD pair whose path includes it.

That is exactly a matrix multiplication:

```
link_loads = demand @ A.T
```

**We read this rather than computing shortest paths ourselves.** `A` records how
Abilene's OSPF actually routed traffic; running Dijkstra would replace a
measurement with an assumption.

In [5]:
# Columns: link_name  od_name  link_idx  demand_idx  fraction
A_long = pd.read_csv(RAW / "A", sep=r"\s+", comment="#",
                     names=["link", "od", "link_idx", "demand_idx", "fraction"])

n_links = link_index["link_idx"].max()        # 54
n_demands = demand_index["demand_idx"].max()  # 144

# Build the dense matrix: rows = links, columns = OD pairs.
# The file indices are 1-based, numpy is 0-based, hence the -1.
A = np.zeros((n_links, n_demands))
A[A_long["link_idx"] - 1, A_long["demand_idx"] - 1] = A_long["fraction"]

# Keep only the rows belonging to internal backbone links.
A_internal = A[internal["link_idx"].to_numpy() - 1, :]

print(f"full routing matrix: {A.shape}  ({(A != 0).sum()} non-zero entries)")
print(f"internal links only: {A_internal.shape}")

# Each column sums to the number of links that OD pair's traffic crosses,
# i.e. its path length in hops. A sanity signal that the matrix is sensible.
hops = A.sum(axis=0)
print(f"path lengths: min {hops.min():.0f} hops, max {hops.max():.0f} hops")

full routing matrix: (54, 144)  (644 non-zero entries)
internal links only: (30, 144)
path lengths: min 2 hops, max 8 hops


## Step 4 — When does each week start, and where are the gaps?

The `X##.gz` files contain **no timestamps of their own** — just 2016 rows of
numbers. `readme.txt` is the only record of which date each file starts on.

Those dates are **not** a clean 7-day sequence. Some weeks are missing entirely,
and one file starts a day *before* the previous one ends. We check this explicitly
rather than assuming the files line up.

In [6]:
# Pull lines that look like:  "  2004-03-01 X01"
week_dates = {}
for line in (RAW / "readme.txt").read_text().splitlines():
    parts = line.split()
    if len(parts) == 2 and parts[1].startswith("X") and parts[1][1:].isdigit():
        week_dates[parts[1]] = pd.Timestamp(parts[0])

weeks = pd.Series(week_dates).sort_index()
print(f"{len(weeks)} weekly files, {weeks.iloc[0]:%Y-%m-%d} to {weeks.iloc[-1]:%Y-%m-%d}")
weeks.head()

24 weekly files, 2004-03-01 to 2004-09-04


X01   2004-03-01
X02   2004-03-08
X03   2004-04-02
X04   2004-04-09
X05   2004-04-22
dtype: datetime64[us]

In [7]:
# Each file is 2016 rows x 5 minutes = exactly 7 days.
# So week N should end exactly where week N+1 begins.
WEEK = pd.Timedelta(days=7)

boundaries = pd.DataFrame({
    "file": weeks.index[:-1],
    "next_file": weeks.index[1:],
    "expected_next_start": (weeks + WEEK).to_numpy()[:-1],
    "actual_next_start": weeks.to_numpy()[1:],
})
boundaries["difference"] = (boundaries["actual_next_start"]
                            - boundaries["expected_next_start"])

# Positive difference = missing data (gap). Negative = the files overlap.
boundaries["kind"] = np.select(
    [boundaries["difference"] > pd.Timedelta(0),
     boundaries["difference"] < pd.Timedelta(0)],
    ["GAP", "OVERLAP"],
    default="contiguous",
)

print(boundaries["kind"].value_counts().to_string())
print("\nThe joins that are not contiguous:")
boundaries[boundaries["kind"] != "contiguous"]

kind
contiguous    18
GAP            4
OVERLAP        1

The joins that are not contiguous:


,file,next_file,expected_next_start,actual_next_start,difference,kind
1,X02,X03,2004-03-15,2004-04-02,18 days,GAP
3,X04,X05,2004-04-16,2004-04-22,6 days,GAP
4,X05,X06,2004-04-29,2004-05-01,2 days,GAP
19,X20,X21,2004-08-14,2004-08-13,-1 days,OVERLAP
20,X21,X22,2004-08-20,2004-08-21,1 days,GAP


## Step 5 — Load the 24 weekly traffic-matrix files

Each row of an `X##.gz` file has **720 numbers = 144 OD pairs × 5 values**:

```
realOD | simpleGravity | simpleTomogravity | generalGravity | generalTomogravity
```

Only the **first** value in each group of five is a real measurement. The other
four are outputs of traffic-matrix *estimation algorithms* that the dataset
authors published alongside it. Using them would mean modelling someone else's
estimates instead of the actual traffic.

Taking every 5th column starting at column 0 (`[:, 0::5]`) keeps only `realOD`.

The units are also unusual: **100 bytes per 5 minutes** (100 because the routers
sampled 1 packet in 100). Converting to kbps:

```
kbps = value x 100 bytes x 8 bits / 300 seconds / 1000
```

In [8]:
frames = []

for name, start in weeks.items():
    with gzip.open(RAW / "traffic-matrices" / f"{name}.gz", "rt") as f:
        raw = np.loadtxt(f)

    measured = raw[:, 0::5]        # every 5th column = the real measurement
    stamps = pd.date_range(start, periods=len(measured), freq="5min")
    frames.append(pd.DataFrame(measured, index=stamps))

    print(f"{name}: {raw.shape[1]} columns -> {measured.shape[1]} OD pairs, "
          f"{measured.shape[0]} timesteps from {start:%Y-%m-%d}")

demand = pd.concat(frames)
print(f"\ncombined: {demand.shape[0]:,} timesteps x {demand.shape[1]} OD pairs")

X01: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-03-01


X02: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-03-08


X03: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-04-02


X04: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-04-09


X05: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-04-22


...

X20: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-08-07


X21: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-08-13


X22: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-08-21


X23: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-08-28


X24: 720 columns -> 144 OD pairs, 2016 timesteps from 2004-09-04

combined: 48,384 timesteps x 144 OD pairs


## Step 6 — Resolve the X20/X21 overlap

`X21` starts on 2004-08-13, which is *inside* `X20`'s last day. That gives us
288 timestamps (one full day at 5-minute spacing) appearing twice.

Before dropping anything, we check **whether the two copies actually agree**.
If they disagreed, that would be a real conflict in the source data and we would
need to decide which to trust. They don't — they're identical — so dropping the
second copy loses nothing.

In [9]:
duplicated = demand.index.duplicated(keep="first")
print(f"duplicate timestamps: {duplicated.sum()}")

# Compare the first and last copy of every duplicated timestamp.
dup_times = demand.index[duplicated].unique()
first_copy = demand.loc[dup_times].groupby(level=0).first()
last_copy = demand.loc[dup_times].groupby(level=0).last()

identical = first_copy.equals(last_copy)
print(f"the two copies are identical: {identical}")
print(f"largest difference between copies: {(first_copy - last_copy).abs().to_numpy().max()}")

demand = demand[~duplicated].sort_index()
print(f"\nafter dropping the duplicates: {demand.shape[0]:,} timesteps")

duplicate timestamps: 288
the two copies are identical: True
largest difference between copies: 0.0

after dropping the duplicates: 48,096 timesteps


## Step 7 — Compute link utilisation

Now the actual calculation, in three lines:

1. convert demand to kbps
2. push demand through the routing matrix to get the load on each link
3. divide by each link's capacity

In [10]:
# 1. raw units (100 bytes / 5 min) -> kbps
KBPS_PER_UNIT = 100 * 8 / 300 / 1000
demand_kbps = demand * KBPS_PER_UNIT

# 2. route the demand onto the links:  loads = demand @ A.T
loads = pd.DataFrame(
    demand_kbps.to_numpy() @ A_internal.T,
    index=demand.index,
    columns=internal["link"],
)

# 3. divide by capacity. `capacity` is a Series indexed by link name, so pandas
#    lines it up with the columns of `loads` automatically.
capacity = links.set_index("link").loc[internal["link"], "capacity_kbps"]
util = loads / capacity * 100

print(f"utilisation table: {util.shape[0]:,} timesteps x {util.shape[1]} links")
util.iloc[:3, :4].round(4)

utilisation table: 48,096 timesteps x 30 links


link,"ATLA-M5,ATLAng","ATLAng,ATLA-M5","ATLAng,HSTNng","ATLAng,IPLSng"
2004-03-01 00:00:00,0.0939,0.2570,1.5574,2.7417
2004-03-01 00:05:00,0.0945,0.3655,1.7694,2.6845
2004-03-01 00:10:00,0.0821,0.3523,1.7600,2.7471


### What does the network actually look like?

This is the finding the whole project rests on: OSPF routing piles traffic onto a
few links while others sit almost empty. That imbalance is what adaptive rerouting
is meant to fix.

In [11]:
summary = pd.DataFrame({
    "capacity_gbps": capacity.to_numpy() / 1e6,
    "mean_util": util.mean(),
    "p95_util": util.quantile(0.95),
    "max_util": util.max(),
}).sort_values("max_util", ascending=False).round(2)

print("busiest links:")
print(summary.head(5).to_string())
print("\nquietest links:")
print(summary.tail(5).to_string())

print(f"\nnetwork-wide mean utilisation: {util.to_numpy().mean():.2f}%")
print(f"highest utilisation anywhere:  {util.to_numpy().max():.2f}%")

busiest links:
               capacity_gbps  mean_util  p95_util  max_util
link                                                       
KSCYng,IPLSng           9.92       5.67      8.63     75.36
IPLSng,CHINng           9.92       6.07      9.08     75.23
DNVRng,KSCYng           9.92       5.44      8.31     75.12
LOSAng,SNVAng           9.92       3.15      4.42     73.49
SNVAng,DNVRng           9.92       2.74      4.07     72.69

quietest links:
                capacity_gbps  mean_util  p95_util  max_util
link                                                        
DNVRng,STTLng            9.92       1.04      1.91      3.73
HSTNng,KSCYng            9.92       0.24      0.55      2.34
SNVAng,STTLng            9.92       0.73      1.07      1.87
ATLA-M5,ATLAng           9.92       0.04      0.11      1.00
ATLAng,ATLA-M5           9.92       0.07      0.20      0.56

network-wide mean utilisation: 2.69%
highest utilisation anywhere:  75.36%


## Step 8 — Sanity checks

Quick pass/fail checks before saving anything.

In [12]:
checks = {
    "no missing values":
        not util.isna().any().any(),
    "no duplicate timestamps":
        not util.index.duplicated().any(),
    "timestamps in order":
        util.index.is_monotonic_increasing,
    "no negative utilisation":
        (util >= 0).all().all(),
    "nothing exceeds capacity":
        (util <= 100).all().all(),
    "30 links":
        util.shape[1] == 30,
    "48,096 timesteps":
        util.shape[0] == 48_096,
}

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

print(f"\n{sum(checks.values())}/{len(checks)} checks passed")

[PASS] no missing values
[PASS] no duplicate timestamps
[PASS] timestamps in order
[PASS] no negative utilisation
[PASS] nothing exceeds capacity
[PASS] 30 links
[PASS] 48,096 timesteps

7/7 checks passed


### Also build the "long" (tidy) shape

The wide table is convenient to look at, but modelling wants one row per
(timestamp, link). `.stack()` does exactly that: it turns each column into rows,
keeping timestamps in order and links in their original order within each
timestamp.

In [13]:
# .stack() gives a Series with a (timestamp, link) MultiIndex.
# rename_axis names the two index levels, then reset_index turns them into columns.
long = (util.stack()
            .rename("utilization_pct")
            .rename_axis(["timestamp", "link"])
            .reset_index())

# Attach the matching load and capacity for each row.
long["load_kbps"] = loads.stack().to_numpy()
long["capacity_kbps"] = long["link"].map(capacity)
long[["src", "dst"]] = long["link"].str.split(",", expand=True)

long = long[["timestamp", "link", "src", "dst",
             "capacity_kbps", "load_kbps", "utilization_pct"]]

print(f"{len(long):,} rows = {util.shape[0]:,} timesteps x {util.shape[1]} links")
long.head(3)

1,442,880 rows = 48,096 timesteps x 30 links


,timestamp,link,src,dst,capacity_kbps,load_kbps,utilization_pct
0,2004-03-01,"ATLA-M5,ATLAng",ATLA-M5,ATLAng,9920000.0,9314.552000,0.093897
1,2004-03-01,"ATLAng,ATLA-M5",ATLAng,ATLA-M5,9920000.0,25490.664000,0.256962
2,2004-03-01,"ATLAng,HSTNng",ATLAng,HSTNng,9920000.0,154496.938667,1.557429


## Step 9 — Verify against the previous pipeline

The earlier version of this project computed the same table with a set of Python
modules. This notebook replaces them, so before we throw those away we check that
the numbers are **exactly** the same — not just similar.

In [14]:
reference_path = PROCESSED / "abilene_link_utilization_wide.parquet"

if reference_path.exists():
    reference = pd.read_parquet(reference_path)
    reference.index = pd.DatetimeIndex(reference.index)

    same_shape = reference.shape == util.shape
    same_index = reference.index.equals(util.index)
    same_columns = list(reference.columns) == list(util.columns)
    # Exact float equality is too strict after a different order of operations;
    # allclose with a very tight tolerance is the honest comparison.
    same_values = np.allclose(reference.to_numpy(), util.to_numpy(),
                              rtol=1e-12, atol=1e-12, equal_nan=True)

    print(f"[{'PASS' if same_shape else 'FAIL'}] shape        {util.shape} vs {reference.shape}")
    print(f"[{'PASS' if same_index else 'FAIL'}] timestamps   identical")
    print(f"[{'PASS' if same_columns else 'FAIL'}] link columns identical")
    print(f"[{'PASS' if same_values else 'FAIL'}] values       match to 1e-12")

    biggest = np.abs(reference.to_numpy() - util.to_numpy()).max()
    print(f"\nlargest absolute difference anywhere: {biggest:.3e}")

    print("\nspot check - CHINng,IPLSng at three timestamps:")
    spot = pd.DataFrame({
        "notebook": util["CHINng,IPLSng"].iloc[[0, 20000, 48095]],
        "old pipeline": reference["CHINng,IPLSng"].iloc[[0, 20000, 48095]],
    })
    display(spot)
else:
    print("no reference file found - skipping comparison")

[PASS] shape        (48096, 30) vs (48096, 30)
[PASS] timestamps   identical
[PASS] link columns identical
[PASS] values       match to 1e-12

largest absolute difference anywhere: 0.000e+00

spot check - CHINng,IPLSng at three timestamps:


,notebook,old pipeline
2004-03-01 00:00:00,5.013087,5.013087
2004-06-04 10:40:00,4.607582,4.607582
2004-09-10 23:55:00,7.973816,7.973816


In [15]:
# Same check for the long/tidy table, including row ORDER.
reference_long_path = PROCESSED / "abilene_link_utilization.parquet"

if reference_long_path.exists():
    reference_long = pd.read_parquet(reference_long_path)

    same_rows = len(reference_long) == len(long)
    same_cols = list(reference_long.columns) == list(long.columns)
    same_order = reference_long["link"].to_numpy()[:100].tolist() == \
                 long["link"].to_numpy()[:100].tolist()
    same_numbers = np.allclose(
        reference_long[["capacity_kbps", "load_kbps", "utilization_pct"]].to_numpy(),
        long[["capacity_kbps", "load_kbps", "utilization_pct"]].to_numpy(),
        rtol=1e-12, atol=1e-12)
    same_stamps = reference_long["timestamp"].equals(long["timestamp"])

    print(f"[{'PASS' if same_rows else 'FAIL'}] row count    {len(long):,}")
    print(f"[{'PASS' if same_cols else 'FAIL'}] columns      {list(long.columns)}")
    print(f"[{'PASS' if same_order else 'FAIL'}] row order    matches (timestamp-major)")
    print(f"[{'PASS' if same_stamps else 'FAIL'}] timestamps   identical")
    print(f"[{'PASS' if same_numbers else 'FAIL'}] values       match to 1e-12")
else:
    print("no reference file found - skipping comparison")

[PASS] row count    1,442,880
[PASS] columns      ['timestamp', 'link', 'src', 'dst', 'capacity_kbps', 'load_kbps', 'utilization_pct']
[PASS] row order    matches (timestamp-major)
[PASS] timestamps   identical
[PASS] values       match to 1e-12


## Step 10 — Save

Two formats, because the next notebook and any modelling want different shapes:

- **wide** — one row per timestamp, one column per link (compact, easy to plot)
- **long** — one row per (timestamp, link) with capacity and load alongside

In [16]:
PROCESSED.mkdir(parents=True, exist_ok=True)

util.to_parquet(PROCESSED / "abilene_link_utilization_wide.parquet")
long.to_parquet(PROCESSED / "abilene_link_utilization.parquet", index=False)

print(f"saved wide: {util.shape[0]:,} timesteps x {util.shape[1]} links")
print(f"saved long: {len(long):,} rows")
print("\nNotebook 02 picks up from the wide table.")

saved wide: 48,096 timesteps x 30 links
saved long: 1,442,880 rows

Notebook 02 picks up from the wide table.
